# Case-Based Reasoning (CBR)
## Tahap 1 - Membangun Case Base
### Domain: Tindak Pidana Perdagangan Orang (TPPO)

Tahap ini bertujuan untuk:

1. Membaca dokumen PDF putusan MA RI
2. Mengekstrak teks putusan
3. Membersihkan teks
4. Menyimpan hasil ke folder data/raw
5. Membuat log proses cleaning

1.Install Library

In [13]:
!pip install pdfplumber


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


2.Import Library

In [14]:
import os
import re
import pdfplumber
from pathlib import Path

3.Setup Folder

In [15]:
PDF_DIR = "../data/raw_pdf"
RAW_DIR = "../data/raw"
LOG_DIR = "../logs"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("Folder siap digunakan")

Folder siap digunakan


4.Cek Jumlah PDF

In [16]:
pdf_files = sorted([
    f for f in os.listdir(PDF_DIR)
    if f.endswith(".pdf")
])

print(f"Jumlah PDF ditemukan: {len(pdf_files)}")

Jumlah PDF ditemukan: 100


5.Preview Nama File

In [17]:
pdf_files[:10]

['putusan_1013_k_pid.sus_2026_20260613104634.pdf',
 'putusan_1013_k_pid.sus_2026_20260622195025.pdf',
 'putusan_103_pid.sus_2025_pn_pts_20260622194809.pdf',
 'putusan_1092_k_pid_2024_20260613154610.pdf',
 'putusan_1153_pk_pid.sus_2024_20260613154739.pdf',
 'putusan_1171_pk_pid.sus_2026_20260613104008.pdf',
 'putusan_11809_k_pid.sus_2025_20260613154817.pdf',
 'putusan_11815_k_pid.sus_2025_20260613154701.pdf',
 'putusan_11_pk_pid.sus_2026_20260613154503.pdf',
 'putusan_12069_k_pid.sus_2025_20260611114339.pdf']

6.Fungsi Ekstraksi PDF

In [18]:
def extract_pdf_text(pdf_path):
    
    full_text = ""

    try:
        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    full_text += page_text + "\n"

    except Exception as e:
        print(f"Gagal membaca {pdf_path}")
        print(e)

    return full_text

7.Fungsi Cleaning

In [19]:
def clean_text(text):

    text = text.lower()

    # hapus teks berulang khas putusan MA
    patterns_to_remove = [
        r'direktori putusan mahkamah agung republik indonesia',
        r'putusan\.mahkamahagung\.go\.id',

        # disclaimer panjang
        r'disclaimer kepaniteraan mahkamah agung republik indonesia.*?waktu kewaktu',

        # footer halaman
        r'dari\s+\d+\s+halaman\s+putusan\s+nomor.*?(?=membaca|menimbang|$)',

        # heading umum
        r'demi keadilan berdasarkan ketuhanan yang maha esa',

        r'mahkamah agung republik indonesia'
    ]

    for pattern in patterns_to_remove:
        text = re.sub(
            pattern,
            ' ',
            text,
            flags=re.IGNORECASE | re.DOTALL
        )

    # hapus deretan huruf tunggal
    text = re.sub(
        r'(?:\b[a-z]\b\s*){5,}',
        ' ',
        text
    )

    # hapus nomor halaman
    text = re.sub(
        r'halaman\s+\d+',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # hapus url
    text = re.sub(
        r'www\.\S+',
        ' ',
        text
    )

    # karakter non-alfanumerik
    text = re.sub(
        r'[^a-z0-9\s.,]',
        ' ',
        text
    )

    # normalisasi spasi
    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    # hapus disclaimer panjang sampai akhir halaman
    text = re.sub(
    r'disclaimer kepaniteraan mahkamah agung republik indonesia.*?dari\s+\d+\s+halaman',
    ' ',
    text,
    flags=re.IGNORECASE | re.DOTALL
    )

    # hapus footer putusan nomor xxx yang muncul di setiap halaman
    text = re.sub(
    r'putusan nomor\s+\d+\s+[a-z\.\/]+\s+\d{4}',
    ' ',
    text,
    flags=re.IGNORECASE
    )

    # hapus pasangan huruf acak seperti:
    # n a, i h, p k, a i l, m b
    text = re.sub(r'\b[a-z]\s+[a-z]\b', ' ', text)

    return text.strip()

8.Proses PDF → TXT + Cleaning

In [20]:
cleaning_log = []

for idx, pdf_file in enumerate(pdf_files, start=1):

    pdf_path = os.path.join(PDF_DIR, pdf_file)

    raw_text = extract_pdf_text(pdf_path)

    cleaned_text = clean_text(raw_text)

    txt_path = os.path.join(
        RAW_DIR,
        f"case_{idx:03d}.txt"
    )

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

    cleaning_log.append(
        f"{pdf_file} -> case_{idx:03d}.txt | {len(cleaned_text)} karakter"
    )

    print(f"[{idx}/{len(pdf_files)}] selesai")

[1/100] selesai
[2/100] selesai
[3/100] selesai
[4/100] selesai
[5/100] selesai
[6/100] selesai
[7/100] selesai
[8/100] selesai
[9/100] selesai
[10/100] selesai
[11/100] selesai
[12/100] selesai
[13/100] selesai
[14/100] selesai
[15/100] selesai
[16/100] selesai
[17/100] selesai
[18/100] selesai
[19/100] selesai
[20/100] selesai
[21/100] selesai
[22/100] selesai
[23/100] selesai
[24/100] selesai
[25/100] selesai
[26/100] selesai
[27/100] selesai
[28/100] selesai
[29/100] selesai
[30/100] selesai
[31/100] selesai
[32/100] selesai
[33/100] selesai
[34/100] selesai
[35/100] selesai
[36/100] selesai
[37/100] selesai
[38/100] selesai
[39/100] selesai
[40/100] selesai
[41/100] selesai
[42/100] selesai
[43/100] selesai
[44/100] selesai
[45/100] selesai
[46/100] selesai
[47/100] selesai
[48/100] selesai
[49/100] selesai
[50/100] selesai
[51/100] selesai
[52/100] selesai
[53/100] selesai
[54/100] selesai
[55/100] selesai
[56/100] selesai
[57/100] selesai
[58/100] selesai
[59/100] selesai
[60/10

9.Simpan Cleaning Log

In [21]:
log_path = os.path.join(
    LOG_DIR,
    "cleaning.log"
)

with open(log_path, "w", encoding="utf-8") as f:

    for row in cleaning_log:
        f.write(row + "\n")

print("Cleaning log berhasil dibuat")

Cleaning log berhasil dibuat


10.Validasi Jumlah File TXT

In [22]:
txt_files = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.endswith(".txt")
])

print("Jumlah file TXT:", len(txt_files))

Jumlah file TXT: 100


11.Validasi Keutuhan Teks

In [23]:
for file in txt_files[:10]:

    path = os.path.join(RAW_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    print(
        file,
        "| jumlah karakter:",
        len(text)
    )

case_001.txt | jumlah karakter: 9333
case_002.txt | jumlah karakter: 9333
case_003.txt | jumlah karakter: 138199
case_004.txt | jumlah karakter: 8511
case_005.txt | jumlah karakter: 6185
case_006.txt | jumlah karakter: 6493
case_007.txt | jumlah karakter: 9340
case_008.txt | jumlah karakter: 8822
case_009.txt | jumlah karakter: 8702
case_010.txt | jumlah karakter: 8744


12.Preview Hasil Cleaning

In [24]:
sample_txt = os.path.join(
    RAW_DIR,
    txt_files[0]
)

with open(sample_txt, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:3000])

gnomor 1013 k pid.sus 2026 memeriksa perkara tindak pidana khusus pada tingkat kasasi yang n adimohonkan oleh penuntut umum pada kejaksaan negeri jakarta selatan, telah memutus perkara terdakwa   nama yety herawatyk alias lis a tempat lahir jakarta   umur tanggal lahir 47 tahun 24 februari 1978   jenis kelamin perempuan   kewarganegaraan indonesia   tempat tinggal jalan kalipasir gg. tembok nomor 24 rt h e011 rw 010, kelurahan kebon sirih, a rkecamatan menteng, jakarta pusat   s agama islam m pekerjaan g swasta e terdakwa tersebut berada dalam tahanan rumah tahanan negara   rutan sejak tanggal 11 november 2024 sampai dengan sekarang   terdakwa diajukan di depan persidangan pengadilan negeri jakarta   selatan karena didakwa dengan dakwaan sebagai berikut n apertama perbuatan terdakwa sebagaimana diatur dan diancam pidana dalam pasal 2 ayat 1 undang undang nomori 21 tahun 2007 h tentang pemberantasan tindak pidana pkerdagangan orang a atau   kedua perbuatan terdakwa sebagaimana diatur da